In [1]:
import torch

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Tesla T4


In [5]:
!pip install transformers datasets sentencepiece accelerate sacrebleu openpyxl -q

In [6]:
import pandas as pd

# Load dataset
df = pd.read_excel("/content/English-Khasi Training Data 2026 (1).xlsx")

# Show first rows
print(df.head())

# Show dataset size
print("Total Rows:", len(df))

                                            English   \
0  Behold , therefore I will bring strangers upon...   
1  Now when Jesus was risen early the first day o...   
2  If men strive , and hurt a woman with child , ...   
3  On the eighth day he sent the people away : an...   
4  And they of Ephraim shall be like a mighty man...   

                                               Khasi  
0  ngan wanrah ki nongshun kiba sniew ban tur ïal...  
1  Hadien ba U Jisu u la mihpat na ka jingïap dan...  
2  Lada ki rangbah kiba ïashoh ki pynmynsaw ïa ka...  
3  Ha ka sngi kaba phra u Solomon u phah noh sha ...  
4  Ki paid Israel kin long kiba khlaiñ kum ki shi...  
Total Rows: 26000


In [7]:
# Rename columns
df.columns = ["english", "khasi"]

# Remove empty rows
df = df.dropna()

# Remove extra spaces
df["english"] = df["english"].astype(str).str.strip()
df["khasi"] = df["khasi"].astype(str).str.strip()

print(df.head())

                                             english  \
0  Behold , therefore I will bring strangers upon...   
1  Now when Jesus was risen early the first day o...   
2  If men strive , and hurt a woman with child , ...   
3  On the eighth day he sent the people away : an...   
4  And they of Ephraim shall be like a mighty man...   

                                               khasi  
0  ngan wanrah ki nongshun kiba sniew ban tur ïal...  
1  Hadien ba U Jisu u la mihpat na ka jingïap dan...  
2  Lada ki rangbah kiba ïashoh ki pynmynsaw ïa ka...  
3  Ha ka sngi kaba phra u Solomon u phah noh sha ...  
4  Ki paid Israel kin long kiba khlaiñ kum ki shi...  


In [8]:
from sklearn.model_selection import train_test_split

# Train 90%
train_df, temp_df = train_test_split(df, test_size=0.1, random_state=42)

# Validation 5%
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("Test:", len(test_df))

Train: 23400
Validation: 1300
Test: 1300


In [9]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)

Dataset({
    features: ['english', 'khasi', '__index_level_0__'],
    num_rows: 23400
})


In [10]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [11]:
model.gradient_checkpointing_enable()

In [12]:
source_lang = "en_XX"

tokenizer.src_lang = source_lang

In [13]:
max_length = 128

def preprocess_function(examples):

    inputs = examples["english"]
    targets = examples["khasi"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [14]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_valid = valid_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/23400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

In [15]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./mbart-khasi",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    weight_decay=0.01,

    save_total_limit=2,

    num_train_epochs=3,

    predict_with_generate=True,

    fp16=True,

    logging_steps=100
)

In [16]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid
)

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.598033,0.577441
2,0.489968,0.520002


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.598033,0.577441
2,0.489968,0.520002
3,0.373953,0.508749


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=35100, training_loss=0.5823013690394214, metrics={'train_runtime': 14578.7261, 'train_samples_per_second': 4.815, 'train_steps_per_second': 2.408, 'total_flos': 1.90165637136384e+16, 'train_loss': 0.5823013690394214, 'epoch': 3.0})

In [ ]:
trainer.save_model("/content/english-khasi-mbart")
tokenizer.save_pretrained("/content/english-khasi-mbart")

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import torch

# Load saved model
model = MBartForConditionalGeneration.from_pretrained("/content/english-khasi-mbart")
tokenizer = MBart50TokenizerFast.from_pretrained("/content/english-khasi-mbart")

# Source language = English
tokenizer.src_lang = "en_XX"

text = "water"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

generated_tokens = model.generate(
    **inputs,
    max_length=128,
    num_beams=4,
    early_stopping=True
)

translation = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Translation:", translation[0])

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Translation: ka um


In [21]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import torch

# Load saved model
model = MBartForConditionalGeneration.from_pretrained("/content/english-khasi-mbart")
tokenizer = MBart50TokenizerFast.from_pretrained("/content/english-khasi-mbart")

# Source language = English
tokenizer.src_lang = "en_XX"

text = "Iam Sajal Samanta. Iam from Techno India University"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

generated_tokens = model.generate(
    **inputs,
    max_length=128,
    num_beams=4,
    early_stopping=True
)

translation = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Translation:", translation[0])

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Translation: Nga dei u Sajal Samanta . Nga dei na ka Techno India University


In [22]:
def translate_text(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    generated_tokens = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return translation[0]

In [23]:
print(translate_text("How are you?"))

Kumno phi long ?


In [24]:
print(translate_text("Artificial intelligence is transforming education and healthcare."))

Ka jingstad ba la thaw da ki briew ka pynkylla ïa ka jingstad bad ka jingïarap .


In [25]:
print(translate_text("Iam Sajal Samanta. Iam from Techno India University."))

Ka kyrteng jong nga ka long Sajal Samanta . Nga dei na ka Techno India University .


In [26]:
print(translate_text("alone"))

tang marwei


In [27]:
!cp -r /content/english-khasi-mbart /content/drive/MyDrive/

In [28]:
!cp -r /content/english-khasi-mbart /content/drive/MyDrive/

In [29]:
!cp -r /content/mbart-khasi/checkpoint-35100 /content/drive/MyDrive/

In [30]:
print(translate_text("I will not go to college tomorrow."))

Ngan ym leit sha ka kamra shuwa .


In [35]:
print(translate_text("Iam Sajal Samanta. Iam from Techno India University.yes no yes no yes no yes no yes no yes no yes no yes no "))

Ka kyrteng jong nga ka long Sajal Samanta . Ka kyrteng jong nga ka long na Techno India University .


In [41]:
print(translate_text("go to hell"))

kin leit sha ka pyrthei kiba ïap
